<a href="https://colab.research.google.com/github/ze-16/Speech-training-agent/blob/main/Inference_test_SAIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import librosa
import torchaudio
import IPython
from torchaudio.utils import _download_asset
import torch

*"Right now I am testing an implementation for ASR from a source, it will be interesting to know how this interacts with the existing wav2vec2 model for emotion recognition, adjustments will be made later ~ 02/03/2026"* **Code shifted from initial notebook**

https://docs.pytorch.org/audio/stable/tutorials/speech_recognition_pipeline_tutorial.html -- source

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.random.manual_seed(0)
print(device)

cpu


In [ ]:
SPEECH_FILE = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav")

100%|██████████| 106k/106k [00:00<00:00, 5.63MB/s]


In [ ]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
print("Sample rate:", bundle.sample_rate)
print("Labels:", bundle.get_labels())

modelASR = bundle.get_model().to(device)

Sample rate: 16000
Labels: ('-', '|', 'E', 'T', 'A', 'O', 'N', 'I', 'H', 'S', 'R', 'D', 'L', 'U', 'M', 'W', 'C', 'F', 'G', 'Y', 'P', 'B', 'V', 'K', "'", 'X', 'J', 'Q', 'Z')
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:05<00:00, 71.1MB/s]


Display.audio is put as a comment to allow github pushes, widgets break the push to github

In [ ]:
#IPython.display.Audio(SPEECH_FILE)

In [ ]:
waveform, sample_rate = torchaudio.load(SPEECH_FILE)
waveform = waveform.to(device)

if sample_rate != bundle.sample_rate:
  waveform = torchaudio.functional.resample(waveform, sample_rate, bundle.sample_rate)

In [ ]:
with torch.inference_mode():
  features, _ = modelASR.extract_features(waveform)

In [ ]:
with torch.inference_mode():
  emission, _ = modelASR(waveform)
print("Class labels:", bundle.get_labels())
print("Shape of emission:", emission.shape)

Class labels: ('-', '|', 'E', 'T', 'A', 'O', 'N', 'I', 'H', 'S', 'R', 'D', 'L', 'U', 'M', 'W', 'C', 'F', 'G', 'Y', 'P', 'B', 'V', 'K', "'", 'X', 'J', 'Q', 'Z')
Shape of emission: torch.Size([1, 169, 29])


In [ ]:
class GreedyCTCDecoder(torch.nn.Module):
  def __init__(self, labels, blank=0):
    super().__init__()
    self.labels = labels
    self.blank = blank

  def forward(self, emission: torch.Tensor) -> str:
    """Given a sequence emission over labels, get the best path string
    Args:
      emission (Tensor): Logit tensors. Shape `[num_seq, num_label]`.

    Returns:
      str: The resulting transcript
      """

    indices = torch.argmax(emission,dim=-1)
    indices = torch.unique_consecutive(indices, dim=-1)
    indices = [i for i in indices if i != self.blank]
    return "".join([self.labels[i] for i in indices])

Display.audio is put as a comment to allow github pushes, widgets break the push to github

In [ ]:
decoder = GreedyCTCDecoder(bundle.get_labels())
transcript = decoder(emission[0])
print(transcript)
#IPython.display.Audio(SPEECH_FILE)

I|HAD|THAT|CURIOSITY|BESIDE|ME|AT|THIS|MOMENT|
